In [2]:
#!pip install torch

In [3]:
#!pip install torchvision

In [4]:
#!pip install kagglehub

In [5]:
## Importing our classes, modules and functions

In [1]:
import torch
import torch.nn as nn
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import DataLoader, Subset


In [2]:
## Downloading the data in root directory

In [3]:
import kagglehub
import shutil
from pathlib import Path

# Download dataset through KaggleHub
path = kagglehub.dataset_download("ambityga/imagenet100")

# Your project directory
project_dir = Path(
    r"C:\Users\Shivanshu Sirohi\Documents\Image_Detection_from_scratch"
)

# Destination
destination = project_dir / "data" / "imagenet100"

# Create data directory
destination.parent.mkdir(parents=True, exist_ok=True)

# Copy dataset into project
shutil.copytree(path, destination, dirs_exist_ok=True)

print("Dataset copied to:")
print(destination)

Dataset copied to:
C:\Users\Shivanshu Sirohi\Documents\Image_Detection_from_scratch\data\imagenet100


In [4]:
## Loadinag the data in a readable format
## Each folder like "Train.X1" contains 25 classes >> 4 train folders 100 classes in total and 100 in val

In [5]:
from pathlib import Path

data_dir = Path(
    r"C:\Users\Shivanshu Sirohi\Documents\Image_Detection_from_scratch\data\imagenet100"
)

train_dir = data_dir / "train.X1"

classes = sorted([x.name for x in train_dir.iterdir() if x.is_dir()])

print("Number of classes in X1:", len(classes))
print("First 10 classes:", classes[:10])

first_class = train_dir / classes[0]

images = list(first_class.iterdir())

print("Images in first class:", len(images))
print("Example:", images[:3])

Number of classes in X1: 25
First 10 classes: ['n01440764', 'n01484850', 'n01494475', 'n01531178', 'n01632777', 'n01665541', 'n01687978', 'n01695060', 'n01749939', 'n01775062']
Images in first class: 1300
Example: [WindowsPath('C:/Users/Shivanshu Sirohi/Documents/Image_Detection_from_scratch/data/imagenet100/train.X1/n01440764/n01440764_10026.JPEG'), WindowsPath('C:/Users/Shivanshu Sirohi/Documents/Image_Detection_from_scratch/data/imagenet100/train.X1/n01440764/n01440764_10027.JPEG'), WindowsPath('C:/Users/Shivanshu Sirohi/Documents/Image_Detection_from_scratch/data/imagenet100/train.X1/n01440764/n01440764_10029.JPEG')]


In [6]:
from pathlib import Path

data_dir = Path(
    r"C:\Users\Shivanshu Sirohi\Documents\Image_Detection_from_scratch\data\imagenet100"
)

for folder_name in ["train.X1", "train.X2", "train.X3", "train.X4", "val.X"]:
    
    folder = data_dir / folder_name
    
    classes = [
        x for x in folder.iterdir()
        if x.is_dir()
    ]
    
    print(f"{folder_name}: {len(classes)} class folders")
    print("  Examples:", [x.name for x in classes[:5]])

train.X1: 25 class folders
  Examples: ['n01440764', 'n01484850', 'n01494475', 'n01531178', 'n01632777']
train.X2: 25 class folders
  Examples: ['n01443537', 'n01514668', 'n01514859', 'n01537544', 'n01592084']
train.X3: 25 class folders
  Examples: ['n01498041', 'n01560419', 'n01582220', 'n01601694', 'n01614925']
train.X4: 25 class folders
  Examples: ['n01491361', 'n01496331', 'n01630670', 'n01667778', 'n01675722']
val.X: 100 class folders
  Examples: ['n01440764', 'n01443537', 'n01484850', 'n01491361', 'n01494475']


In [7]:
import json

labels_path = data_dir / "Labels.json"

with open(labels_path, "r") as f:
    labels = json.load(f)

print(type(labels))
print(list(labels.items())[:5] if isinstance(labels, dict) else labels[:5])

<class 'dict'>
[('n01968897', 'chambered nautilus, pearly nautilus, nautilus'), ('n01770081', 'harvestman, daddy longlegs, Phalangium opilio'), ('n01818515', 'macaw'), ('n02011460', 'bittern'), ('n01496331', 'electric ray, crampfish, numbfish, torpedo')]


In [8]:
## Building the class mapping

In [9]:
from pathlib import Path

data_dir = Path(
    r"C:\Users\Shivanshu Sirohi\Documents\Image_Detection_from_scratch\data\imagenet100"
)

train_folders = [
    data_dir/"train.X1",
    data_dir/"train.X2",
    data_dir/"train.X3",
    data_dir/"train.X4"
]

# get all class names
class_names = sorted({
    class_dir.name
    for folder in train_folders
    for class_dir in folder.iterdir()
    if class_dir.is_dir()
})

print("Number of classes:", len(class_names))
print("First 10:", class_names[:10])

Number of classes: 100
First 10: ['n01440764', 'n01443537', 'n01484850', 'n01491361', 'n01494475', 'n01496331', 'n01498041', 'n01514668', 'n01514859', 'n01531178']


In [10]:
class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(class_names)
}

print(list(class_to_idx.items())[:10])

[('n01440764', 0), ('n01443537', 1), ('n01484850', 2), ('n01491361', 3), ('n01494475', 4), ('n01496331', 5), ('n01498041', 6), ('n01514668', 7), ('n01514859', 8), ('n01531178', 9)]


In [11]:
## Budilding our custom PyTorch Dataset
## "Dataset" stores the samples and their corresponding labels, "DataLoader" wraps an iterable around the dataset tp enable easy access to the samples

In [12]:
from torch.utils.data import Dataset
from PIL import Image

class ImageNet100Dataset(Dataset):

    def __init__(self, folders, class_to_idx, transform=None):

        self.samples = []
        # Empty list that will eventually contain:
        # (image_path, label)
        #
        # We are NOT loading images here.
        # We are only recording where each image is stored.

        self.transform = transform
        # Whatever transform we provide will be stored here.
        # We will use this later for resizing, normalization,
        # augmentation, etc.


        # Go through each training shard
        for folder in folders:

            # Go through each class folder inside the shard
            for class_dir in folder.iterdir():

                # Make sure this is actually a directory
                if not class_dir.is_dir():
                    continue

                # Get the class name
                class_name = class_dir.name

                # Convert class name into integer label
                label = class_to_idx[class_name]


                # Go through every image inside this class folder
                for image_path in class_dir.iterdir():

                    # Make sure it is a file
                    if image_path.is_file():

                        # Store the path and its corresponding label
                        self.samples.append(
                            (image_path, label)
                        )


    def __len__(self):

        # Return the total number of images
        return len(self.samples)


    def __getitem__(self, index):

        # Get the path and label for this particular sample
        image_path, label = self.samples[index]

        # Open the image
        image = Image.open(image_path).convert("RGB")

        # Apply transformations if provided
        if self.transform is not None:
            image = self.transform(image)

        # Return image and label
        return image, label

In [13]:
## Now instantiating the above class

train_dataset = ImageNet100Dataset(
    folders = train_folders,
    class_to_idx=class_to_idx
)

print("Number of images:", len(train_dataset))
image, label = train_dataset[0]
print("Image type:", type(image))                         # currently, this will be a PIL image and not a tensor, we will convert it using transform later
print("Image size:", image.size)
print("Label:", label)

Number of images: 130000
Image type: <class 'PIL.Image.Image'>
Image size: (250, 250)
Label: 0


In [14]:
## Transforming the datasets
train_transform = v2.Compose([
    v2.RandomResizedCrop(224),                    # ImageNet images have different dimensions, we need a fixed input, first random crop and then resize to a 224 x 224
    v2.RandomHorizontalFlip(),
    v2.ToImage(),                                 # PIL [h, w ,c] becomes Tensor [c, h ,w]
    v2.ToDtype(scale=True, dtype=torch.float32),
    v2.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

In [15]:
train_dataset = ImageNet100Dataset(
    folders = train_folders,
    class_to_idx= class_to_idx,
    transform = train_transform
)

In [16]:
image, label = train_dataset[0]

print(type(image))
print(image.shape)
print(label)

<class 'torchvision.tv_tensors._image.Image'>
torch.Size([3, 224, 224])
0


In [17]:
val_transform = v2.Compose([             # this will not be random but deterministic
    v2.Resize(224),
    v2.CenterCrop(224),
    v2.ToImage(),
    v2.ToDtype(dtype=torch.float32, scale = True),
    v2.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

In [18]:
val_folders = [
    data_dir / "val.X"
]

val_dataset = ImageNet100Dataset(
    folders=val_folders,
    class_to_idx=class_to_idx,
    transform=val_transform
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Training images: 130000
Validation images: 5000


In [19]:
## Taking the Dataset Objects and wrapping them in our DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size = 64,
    shuffle = True, 
    num_workers = 0,                                 # 4 CPU processes load the image in parallel
    pin_memory = True                                # can speed CPU > GPU transfers when using CUDA
)

val_loader = DataLoader(
    val_dataset,
    batch_size = 64,
    shuffle = False,
    num_workers = 0,
    pin_memory = True
)

In [20]:
images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)

Images: torch.Size([64, 3, 224, 224])
Labels: torch.Size([64])


In [21]:
## Building Basic Block (Residual Block)

class BasicBlock(nn.Module):                                        # creating our own residual block which inherits from nn.Module
    expansion = 1                                                   # if we want to to later expand the output channels 

    def __init__(self, in_channels, out_channels, stride=1):        # how many channels coming into the block, and how many are going out
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size = 3,
            stride=stride,
            padding=1,                                              # allows preservation of spatial dimensions after a 3 x 3 convolution
            bias=False                                              # immediately after a conv layer we will use BatchNorm, which makes a separate bias term unnecessary during this convolution
        )

        self.bn1 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(
            out_channels,                                           # the in and out channels are same here
            out_channels, 
            kernel_size = 3,
            stride=1,                                               # we are fixing the stride at 1 because we don't want any downsampling of dimensions here
            padding=1,
            bias = False
        )

        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()                             # create an empty sequential module, because if shapes match, input can be directly added to output

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size = 1,
                    stride = stride, 
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)                                 # if input shape matches output this will return x

        out = self.conv1(x) 
        out = self.bn1(out) 
        out = self.relu(out) 

        out = self.conv2(out) 
        out = self.bn2(out) 
        out += identity 
        out = self.relu(out) 
        return out

In [22]:

## Building the ResNet 18

class ResNet18(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()

        self.in_channels = 64                              # this is an important bookkeeping variable i.e. it tells "_make_layer" how many channels are coming into our next residual layer

        ## STEM = The initial processing before the residual blocks
        self.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=7,
            stride = 2,
            padding = 3,
            bias = False
        )

        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.maxpool = nn.MaxPool2d(
            kernel_size = 3,
            stride = 2,
            padding = 1
        )

        ## Residual Layers

        self.layer1 = self._make_layer(64, 2,stride=1)     # each layer has 2 blocks
        self.layer2 = self._make_layer(128,2,stride=2)
        self.layer3 = self._make_layer(256,2,stride=2)
        self.layer4 = self._make_layer(512,2,stride=2)     # total 8 blocks and 8 * 2 = 16 convolutions and the stem makes it 17

        ## Classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))         # takes each of the 512 feature map of dimension 7x7 and reduces it to 1x1

        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, out_channels, blocks, stride):
        layers = []                                        # we are going to put our basic blocks in this list

        layers.append(                                     # our first residual block
            BasicBlock(
                self.in_channels,
                out_channels,
                stride
            )
        )

        self.in_channels = out_channels                    # update in_channels because the new in_channels is 128

        for _ in range(1, blocks):                         # create the remaining blocks, start at 1 because we already created the first one above
            layers.append(
                BasicBlock(
                    self.in_channels,                      # no stride specified and thus will use the basic block default stride=1
                    out_channels
                )
            )

        return nn.Sequential(*layers)                      # PyTorch will execute blocks in sequence

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)

        x = torch.flatten(x, 1)

        x = self.fc(x)

        return x

In [23]:
## Instantiate the model and test it's output
model = ResNet18(num_classes=100)
x = torch.randn(4,3,224,224)
output = model(x)
print(output.shape)

torch.Size([4, 100])


In [24]:
## Checking how large my model is
total_params = sum(
    p.numel() for p in model.parameters()           # for a standard resnet with 1000 classes the param count should be around 11.7M
)

print(f"Total parameters: {total_params:,}")

Total parameters: 11,227,812


In [30]:
model = ResNet18(num_classes=100)

## Loss function
criterion = nn.CrossEntropyLoss()

## Optimizer
optimizer = torch.optim.SGD(
    model.parameters(),
    lr = 0.1,
    momentum = 0.9,
    weight_decay = 1e-4
)

## Learning rate scheduler
scheduler = torch.optim.lr_scheduler.StepLR(        # scheduler will not change the gradient itself but rather the step size applied to it
    optimizer, 
    step_size = 3,                                            # after every 30 epochs, our scheduler will change the LR
    gamma = 0.1                                                # to 0.1 of the current rate, a constant scheduling pattern
    )

In [31]:
## training loop
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device", device)
model = model.to(device)
num_epochs = 10
best_val_acc = 0.0

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)

        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_loss = running_train_loss/total_train
    train_acc = 100 * correct_train / total_train

    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()


    # Calculate epoch validation metrics
    val_loss = running_val_loss / total_val
    val_acc = 100 * correct_val / total_val

    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]


    # ==========================================
    # Save best model
    # ==========================================

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "resnet18_imagenet100_best.pth"
        )


    # ==========================================
    # Print epoch results
    # ==========================================

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.2f}% "
        f"LR: {current_lr:.5f}"
    )






Using device cuda
Epoch [1/10] Train Loss: 3.8611 Train Acc: 10.51% Val Loss: 3.2571 Val Acc: 20.96% LR: 0.10000
Epoch [2/10] Train Loss: 3.0367 Train Acc: 25.39% Val Loss: 2.8388 Val Acc: 29.40% LR: 0.10000
Epoch [3/10] Train Loss: 2.5736 Train Acc: 35.32% Val Loss: 2.5690 Val Acc: 36.06% LR: 0.01000
Epoch [4/10] Train Loss: 2.0226 Train Acc: 47.80% Val Loss: 1.8639 Val Acc: 50.12% LR: 0.01000
Epoch [5/10] Train Loss: 1.8796 Train Acc: 51.04% Val Loss: 1.7750 Val Acc: 53.18% LR: 0.01000
Epoch [6/10] Train Loss: 1.7873 Train Acc: 53.27% Val Loss: 1.7393 Val Acc: 53.52% LR: 0.00100
Epoch [7/10] Train Loss: 1.6520 Train Acc: 56.35% Val Loss: 1.6139 Val Acc: 56.66% LR: 0.00100
Epoch [8/10] Train Loss: 1.6261 Train Acc: 56.90% Val Loss: 1.5974 Val Acc: 56.94% LR: 0.00100
Epoch [9/10] Train Loss: 1.6029 Train Acc: 57.70% Val Loss: 1.5857 Val Acc: 57.40% LR: 0.00010
Epoch [10/10] Train Loss: 1.5884 Train Acc: 58.08% Val Loss: 1.5804 Val Acc: 57.58% LR: 0.00010


In [27]:
import time

model.train()

start = time.time()

for batch_idx, (images, labels) in enumerate(train_loader):

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    if batch_idx == 99:
        break

elapsed = time.time() - start

print(f"100 batches took: {elapsed / 60:.2f} minutes")
print(f"Average per batch: {elapsed / 100:.2f} seconds")
print(f"Estimated training time per epoch: {(elapsed / 100) * len(train_loader) / 60:.2f} minutes")

100 batches took: 1.13 minutes
Average per batch: 0.68 seconds
Estimated training time per epoch: 22.91 minutes
